In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
import os, sys, time, json, datetime, argparse
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from PIL import Image

import segmentation_models_pytorch as smp
import albumentations as A

# Acces au schema NPZ partage du projet
PROJECT_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from shared.npz_schema import load_unified_npz


In [ ]:
from dataclasses import dataclass, field

@dataclass
class TrainConfig:
    # Detection de points d'intersection (5mm) via NPZ -- version PATCHES 256x256.
    # Au lieu de redimensionner toute l'image a 1024x1024 (ce qui ecrase le pas de
    # grille sur les formats denses type 12x1), on entraine sur des crops 256x256
    # pris a RESOLUTION NATIVE. Le pas de grille reste intact -> intersections nettes.

    # Chemins
    project_root: str = r"C:\Users\v\Desktop\ECGPerturb-main\data"
    image_dir: str = ""
    npz_dir:   str = ""
    output_dir: str = ""

    # Cible : intersections de la grille 5mm (cle NPZ)
    npz_key: str = "grid_major_5mm"
    point_radius: int = 3   # rayon (px) des disques dessines pour les ground truth

    # --- Patches ---
    patch_size: int = 256              # taille des crops (resolution native)
    train_patches_per_image: int = 8   # nb de crops aleatoires tires par image / epoch
    val_patches_per_image:   int = 4   # nb de crops deterministes par image (metrique stable)
    tile_overlap: int = 32             # recouvrement des tuiles a l'inference

    # Architecture
    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"
    in_channels: int = 3
    num_classes: int = 1

    # Entrainement
    batch_size: int = 16   # patches petits -> batch plus large possible
    num_epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay:  float = 1e-5
    num_workers: int = 0
    pin_memory:  bool = True

    # Loss & scheduler
    loss_type: str = "bce_dice"
    bce_weight: float = 0.5
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    early_stop_patience: int = 15

    # Split par prefixe de nom de fichier
    train_sources: list = field(default_factory=lambda: ["ECG_031", "ECG_032"])
    val_sources:   list = field(default_factory=lambda: ["ECG_033"])

    device: str = ""
    seed: int = 42
    save_every_n_epochs:    int = 10
    log_images_every_n_epochs: int = 5

    def __post_init__(self):
        if not self.image_dir:
            self.image_dir = os.path.join(self.project_root, "output_augmentation", "images")
        if not self.npz_dir:
            self.npz_dir = os.path.join(self.project_root, "output_augmentation", "npz")
        if not self.output_dir:
            self.output_dir = os.path.join(self.project_root, "training", "runs_npz_patches256")
        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            print(f"[OK] GPU detecte : {torch.cuda.get_device_name(0)}")
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"   VRAM: {vram:.1f} GB")
        else:
            print("[!] Mode CPU actif - lent.")

cfg = TrainConfig()


In [ ]:
class ECGNpzPatchDataset(Dataset):
    # Dataset PATCH-BASED : crops patch_size x patch_size pris a RESOLUTION NATIVE.
    # Aucun downscale global -> le pas de grille reste intact (cle pour formats denses).
    #   mode="train" : crops aleatoires, biaises vers les zones contenant des points.
    #   mode="val"   : crops deterministes (seed = idx) -> metrique stable entre epochs.

    def __init__(self, image_dir, npz_dir, npz_key="grid_major_5mm",
                 source_prefixes=None, patch_size=256, point_radius=3,
                 mode="train", patches_per_image=8):
        self.image_dir = image_dir
        self.npz_dir   = npz_dir
        self.npz_key   = npz_key
        self.P = patch_size
        self.point_radius = point_radius
        self.mode = mode
        self.patches_per_image = patches_per_image

        self.samples = []
        for fname in sorted(os.listdir(image_dir)):
            if not fname.endswith(".webp"):
                continue
            if source_prefixes and not any(fname.startswith(p) for p in source_prefixes):
                continue
            stem = os.path.splitext(fname)[0]
            npz_path = os.path.join(npz_dir, stem + ".npz")
            if os.path.exists(npz_path):
                self.samples.append({
                    "image_path": os.path.join(image_dir, fname),
                    "npz_path":   npz_path,
                    "stem":       stem,
                })

        # Augmentations couleur + flips (geometrie coherente dans le patch)
        if mode == "train":
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.ColorJitter(brightness=0.15, contrast=0.15,
                              saturation=0.1, hue=0.05, p=0.5),
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None

        print(f"  -> {len(self.samples)} images ({mode}, key:{npz_key}, "
              f"sources:{source_prefixes or 'toutes'}, {patches_per_image} patches/img "
              f"= {len(self.samples)*patches_per_image} patches)")

    def __len__(self):
        return len(self.samples) * self.patches_per_image

    @staticmethod
    def _draw_points_patch(pts_xy, x0, y0, P, radius):
        # Disques dessines uniquement dans le patch (points decales de -x0,-y0).
        mask = np.zeros((P, P), dtype=np.uint8)
        if len(pts_xy) == 0:
            return mask
        xs = np.round(pts_xy[:, 0]).astype(int) - x0
        ys = np.round(pts_xy[:, 1]).astype(int) - y0
        keep = (xs >= -radius) & (xs < P + radius) & (ys >= -radius) & (ys < P + radius)
        xs, ys = xs[keep], ys[keep]
        for dx in range(-radius, radius + 1):
            for dy in range(-radius, radius + 1):
                if dx*dx + dy*dy <= radius*radius:
                    xx, yy = xs + dx, ys + dy
                    v = (xx >= 0) & (xx < P) & (yy >= 0) & (yy < P)
                    mask[yy[v], xx[v]] = 255
        return mask

    def _sample_crop(self, Wn, Hn, pts, rng):
        # Prefere un crop contenant >=1 point (jusqu'a 10 essais).
        P = self.P
        maxx, maxy = max(Wn - P, 0), max(Hn - P, 0)
        last = (0, 0)
        for _ in range(10):
            x0 = rng.randint(0, maxx + 1) if maxx > 0 else 0
            y0 = rng.randint(0, maxy + 1) if maxy > 0 else 0
            last = (x0, y0)
            if len(pts) == 0:
                return x0, y0
            inside = ((pts[:, 0] >= x0) & (pts[:, 0] < x0 + P) &
                      (pts[:, 1] >= y0) & (pts[:, 1] < y0 + P)).any()
            if inside:
                return x0, y0
        return last

    def __getitem__(self, idx):
        s = self.samples[idx % len(self.samples)]

        img = Image.open(s["image_path"]).convert("RGB")
        Wn, Hn = img.size
        data = load_unified_npz(s["npz_path"])
        pts = data.get(self.npz_key, np.empty((0, 2)))

        # train : aleatoire ; val : seed=idx (crop stable d'une epoch a l'autre)
        rng = np.random.RandomState() if self.mode == "train" else np.random.RandomState(idx)
        x0, y0 = self._sample_crop(Wn, Hn, pts, rng)

        patch = img.crop((x0, y0, x0 + self.P, y0 + self.P))
        if patch.size != (self.P, self.P):  # padding si image < patch (rare)
            padded = Image.new("RGB", (self.P, self.P))
            padded.paste(patch, (0, 0))
            patch = padded
        img_np = np.array(patch, dtype=np.float32) / 255.0

        mask_np = self._draw_points_patch(pts, x0, y0, self.P,
                                          self.point_radius).astype(np.float32) / 255.0

        if self.transform:
            t = self.transform(image=img_np, mask=mask_np)
            img_np, mask_np = t["image"], t["mask"]

        img_tensor  = torch.from_numpy(img_np).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        return img_tensor, mask_tensor


In [ ]:
# ═══════════════ Loss ═══════════════
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        ps = torch.sigmoid(pred)
        inter = (ps * target).sum(dim=(2, 3))
        union = ps.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce, self.dice, self.bce_weight = nn.BCEWithLogitsLoss(), DiceLoss(), bce_weight
    def forward(self, pred, target):
        return self.bce_weight * self.bce(pred, target) + (1 - self.bce_weight) * self.dice(pred, target)

def get_loss(loss_type, bce_weight=0.5):
    return {"bce": nn.BCEWithLogitsLoss(),
            "dice": DiceLoss(),
            "bce_dice": BCEDiceLoss(bce_weight)}[loss_type]


# ═══════════════ Metrics ═══════════════
def compute_metrics(pred, target, threshold=0.5):
    with torch.no_grad():
        pb = (torch.sigmoid(pred) > threshold).float()
        inter = (pb * target).sum(dim=(2, 3))
        ps = pb.sum(dim=(2, 3)); ts = target.sum(dim=(2, 3))
        dice = (2*inter + 1e-6) / (ps + ts + 1e-6)
        iou  = (inter + 1e-6) / (ps + ts - inter + 1e-6)
        rec  = (inter + 1e-6) / (ts + 1e-6)
        prec = (inter + 1e-6) / (ps + 1e-6)
    return {"dice": dice.mean().item(), "iou": iou.mean().item(),
            "precision": prec.mean().item(), "recall": rec.mean().item()}


# ═══════════════ Visualisation ═══════════════
def save_prediction_grid(images, masks_true, masks_pred, save_path, n=4):
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = axes[np.newaxis, :]
    for i in range(n):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        mt  = masks_true[i, 0].cpu().numpy()
        mp  = (torch.sigmoid(masks_pred[i, 0]).cpu().numpy() > 0.5).astype(float)
        axes[i, 0].imshow(img);                  axes[i, 0].set_title("Patch", fontsize=9); axes[i, 0].axis("off")
        axes[i, 1].imshow(mt, cmap="gray", vmin=0, vmax=1); axes[i, 1].set_title("Points GT (5mm)", fontsize=9); axes[i, 1].axis("off")
        axes[i, 2].imshow(mp, cmap="gray", vmin=0, vmax=1); axes[i, 2].set_title("Points predits", fontsize=9); axes[i, 2].axis("off")
        ov = img.copy()
        tp = (mp > 0.5) & (mt > 0.5); fp = (mp > 0.5) & (mt < 0.5); fn = (mp < 0.5) & (mt > 0.5)
        ov[tp] = [0, 1, 0]; ov[fp] = [1, 0, 0]; ov[fn] = [0, 0, 1]
        axes[i, 3].imshow(ov); axes[i, 3].set_title("Overlay (V=TP, R=FP, B=FN)", fontsize=9); axes[i, 3].axis("off")
    plt.tight_layout(); plt.savefig(save_path, dpi=100, bbox_inches="tight"); plt.close()


# ═══════════════ Boucles d'entrainement ═══════════════
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{m['dice']:.3f}")
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}


def _plot_training_curves(history, save_path):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    e = range(1, len(history["train_loss"]) + 1)
    a1.plot(e, history["train_loss"], "b-", label="Train")
    a1.plot(e, history["val_loss"],   "r-", label="Val")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.set_title("Loss"); a1.legend(); a1.grid(True, alpha=0.3)
    a2.plot(e, history["train_dice"], "b-",  label="Train Dice")
    a2.plot(e, history["val_dice"],   "r-",  label="Val Dice")
    a2.plot(e, history["train_iou"],  "b--", alpha=0.5, label="Train IoU")
    a2.plot(e, history["val_iou"],    "r--", alpha=0.5, label="Val IoU")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Score"); a2.set_title("Dice & IoU"); a2.legend(); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close()


def train(cfg: TrainConfig):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(cfg.output_dir, f"run_{timestamp}")
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visualizations"), exist_ok=True)

    cfg_dict = {k: (v if isinstance(v, (int, float, bool, list, type(None))) else str(v))
                for k, v in cfg.__dict__.items()}
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg_dict, f, indent=2)

    print(f"\n{'='*60}\n  U-Net PATCHES {cfg.patch_size}x{cfg.patch_size} -- Detection points intersections (NPZ)\n{'='*60}")
    print(f"  Device      : {cfg.device}")
    print(f"  Patch       : {cfg.patch_size}x{cfg.patch_size}  (resolution native, pas de downscale)")
    print(f"  Cible NPZ   : {cfg.npz_key}  (point_radius={cfg.point_radius}px)")
    print(f"  Encoder     : {cfg.encoder_name}")
    print(f"  Loss        : {cfg.loss_type}")
    print(f"  Batch size  : {cfg.batch_size}")
    print(f"  Epochs      : {cfg.num_epochs}")
    print(f"  Output      : {run_dir}\n{'='*60}\n")

    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device = torch.device(cfg.device)

    print("[*] Chargement des donnees...")
    print(f"  Images: {cfg.image_dir}")
    print(f"  NPZ   : {cfg.npz_dir}")

    print(f"\n  Train (sources: {cfg.train_sources}):")
    train_ds = ECGNpzPatchDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                                  cfg.train_sources, cfg.patch_size, cfg.point_radius,
                                  mode="train", patches_per_image=cfg.train_patches_per_image)
    print(f"  Val   (sources: {cfg.val_sources}):")
    val_ds = ECGNpzPatchDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                                cfg.val_sources, cfg.patch_size, cfg.point_radius,
                                mode="val", patches_per_image=cfg.val_patches_per_image)

    if len(train_ds) == 0:
        print("\n[ERREUR] Aucune donnee d'entrainement trouvee."); return

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

    print("\n[*] Construction du modele...")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=cfg.encoder_weights,
                     in_channels=cfg.in_channels, classes=cfg.num_classes, activation=None).to(device)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Parametres: {total:,}")

    criterion = get_loss(cfg.loss_type, cfg.bce_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.scheduler_patience, factor=cfg.scheduler_factor)

    history = {k: [] for k in ["train_loss", "val_loss", "train_dice", "val_dice", "train_iou", "val_iou", "lr"]}
    best_val_dice, no_improve = 0, 0

    print(f"\n>>> Debut de l'entrainement ({cfg.num_epochs} epochs)...\n")
    t_start = time.time()

    for epoch in range(1, cfg.num_epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]

        tl, tm = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, vm = validate(model, val_loader, criterion, device)
        scheduler.step(vl)

        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_dice"].append(tm["dice"]); history["val_dice"].append(vm["dice"])
        history["train_iou"].append(tm["iou"]);   history["val_iou"].append(vm["iou"])
        history["lr"].append(lr)

        print(f"Epoch {epoch:3d}/{cfg.num_epochs} | "
              f"Train Loss: {tl:.4f}  Dice: {tm['dice']:.3f} | "
              f"Val Loss: {vl:.4f}  Dice: {vm['dice']:.3f}  IoU: {vm['iou']:.3f} | "
              f"LR: {lr:.1e} | {time.time()-t0:.1f}s")

        if vm["dice"] > best_val_dice:
            best_val_dice = vm["dice"]; no_improve = 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": best_val_dice, "config": cfg_dict},
                       os.path.join(run_dir, "checkpoints", "best_model.pth"))
            print(f"  [BEST] Nouveau meilleur modele (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1

        if epoch % cfg.save_every_n_epochs == 0:
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": vm["dice"]},
                       os.path.join(run_dir, "checkpoints", f"checkpoint_epoch{epoch:03d}.pth"))

        if epoch % cfg.log_images_every_n_epochs == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                si, sm = next(iter(val_loader))
                sp = model(si.to(device)).cpu()
                save_prediction_grid(si, sm, sp,
                    os.path.join(run_dir, "visualizations", f"epoch_{epoch:03d}.png"))

        if no_improve >= cfg.early_stop_patience:
            print(f"\n[STOP] Early stopping apres {cfg.early_stop_patience} epochs sans amelioration"); break

    total_time = time.time() - t_start
    print(f"\n{'='*60}\n  Entrainement termine en {total_time/60:.1f} minutes")
    print(f"  Meilleur Val Dice: {best_val_dice:.4f}")
    print(f"  Resultats dans: {run_dir}\n{'='*60}")
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    _plot_training_curves(history, os.path.join(run_dir, "training_curves.png"))
    return run_dir


In [ ]:
# Lancement de l'entrainement
train(cfg)


In [ ]:
import glob
from IPython.display import display

runs_dir = cfg.output_dir
run_dirs = sorted(glob.glob(os.path.join(runs_dir, "run_*")))
if not run_dirs:
    raise FileNotFoundError(f"Aucun run trouve dans {runs_dir}")
run_dir = run_dirs[-1]
print(f"Run selectionne : {run_dir}")

best_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
ckpt_list = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "*.pth")))
checkpoint_path = best_path if os.path.exists(best_path) else (ckpt_list[-1] if ckpt_list else None)
if checkpoint_path is None:
    raise FileNotFoundError("Aucun checkpoint trouve")
print(f"Checkpoint : {checkpoint_path}")

DEVICE = cfg.device
model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                 in_channels=cfg.in_channels, classes=cfg.num_classes)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"Modele charge - epoch {checkpoint['epoch']} | Val Dice : {checkpoint.get('val_dice', 'N/A')}")


In [ ]:
image_dir = cfg.image_dir
val_images   = sorted([f for f in os.listdir(image_dir)
                       if f.startswith("ECG_033") and f.endswith(".webp")])
train_images = sorted([f for f in os.listdir(image_dir)
                       if (f.startswith("ECG_031") or f.startswith("ECG_032"))
                       and f.endswith(".webp")])
print(f"Train: {len(train_images)} | Val: {len(val_images)}")


In [ ]:
%matplotlib inline
SET   = "val"   # "val" ou "train"
INDEX = 34

image_list = val_images if SET == "val" else train_images
if INDEX >= len(image_list):
    raise IndexError(f"INDEX={INDEX} hors limites - {len(image_list)} images dans '{SET}'")

sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
npz_path    = os.path.join(cfg.npz_dir, sample_file.replace(".webp", ".npz"))

print(f"Image : {sample_file}")
if not os.path.exists(npz_path):
    raise FileNotFoundError(f"NPZ introuvable : {npz_path}")


def _draw_points_full(pts_xy, H, W, radius):
    # Mask GT pleine resolution (disques aux intersections).
    mask = np.zeros((H, W), dtype=np.uint8)
    if len(pts_xy) == 0:
        return mask
    xs = np.round(pts_xy[:, 0]).astype(int)
    ys = np.round(pts_xy[:, 1]).astype(int)
    valid = (xs >= 0) & (xs < W) & (ys >= 0) & (ys < H)
    xs, ys = xs[valid], ys[valid]
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            if dx*dx + dy*dy <= radius*radius:
                mask[np.clip(ys + dy, 0, H - 1), np.clip(xs + dx, 0, W - 1)] = 255
    return mask


def predict_full_image(model, img_pil, patch_size, overlap, device):
    # Inference par tuiles a RESOLUTION NATIVE puis recollage (moyenne sur recouvrements).
    W, H = img_pil.size
    img_np = np.array(img_pil, dtype=np.float32) / 255.0
    acc = np.zeros((H, W), dtype=np.float32)
    cnt = np.zeros((H, W), dtype=np.float32)
    step = max(patch_size - overlap, 1)

    xs = list(range(0, max(W - patch_size, 0) + 1, step)) or [0]
    ys = list(range(0, max(H - patch_size, 0) + 1, step)) or [0]
    if W > patch_size and xs[-1] != W - patch_size: xs.append(W - patch_size)
    if H > patch_size and ys[-1] != H - patch_size: ys.append(H - patch_size)

    model.eval()
    with torch.no_grad():
        for y0 in ys:
            for x0 in xs:
                patch = img_np[y0:y0+patch_size, x0:x0+patch_size]
                ph, pw = patch.shape[:2]
                if (ph, pw) != (patch_size, patch_size):
                    pad = np.zeros((patch_size, patch_size, 3), dtype=np.float32)
                    pad[:ph, :pw] = patch
                    patch = pad
                t = torch.from_numpy(patch).permute(2, 0, 1).unsqueeze(0).float().to(device)
                p = torch.sigmoid(model(t)).squeeze().cpu().numpy()
                acc[y0:y0+ph, x0:x0+pw] += p[:ph, :pw]
                cnt[y0:y0+ph, x0:x0+pw] += 1
    cnt[cnt == 0] = 1
    return acc / cnt


# === Inference plein-image par tuiles ===
img_pil = Image.open(sample_path).convert("RGB")
Wn, Hn = img_pil.size
pred_full = predict_full_image(model, img_pil, cfg.patch_size, cfg.tile_overlap, DEVICE)
pred_binary = (pred_full > 0.5).astype(np.float32)

# === Mask GT plein-resolution ===
data = load_unified_npz(npz_path)
pts  = data.get(cfg.npz_key, np.empty((0, 2)))
mask_np = _draw_points_full(pts, Hn, Wn, cfg.point_radius).astype(np.float32) / 255.0

inter = (pred_binary * mask_np).sum()
dice  = (2 * inter) / (pred_binary.sum() + mask_np.sum() + 1e-8)
print(f"Resolution native : {Wn}x{Hn}  ({len(pts)} points GT)")
print(f"Max: {pred_full.max():.4f} | Moyenne: {pred_full.mean():.4f}")
print(f"Dice (plein-image, tuiles) : {dice:.4f}")

# === Overlay (sur image native, downscale pour affichage) ===
img_disp = np.array(img_pil, dtype=np.float32) / 255.0
overlay = img_disp.copy()
tp = (pred_binary > 0.5) & (mask_np > 0.5)
fp = (pred_binary > 0.5) & (mask_np < 0.5)
fn = (pred_binary < 0.5) & (mask_np > 0.5)
overlay[tp] = [0.0, 1.0, 0.0]
overlay[fp] = [1.0, 0.0, 0.0]
overlay[fn] = [0.0, 0.0, 1.0]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle(f"[{SET.upper()}] {sample_file} - Dice: {dice:.4f}  (patches {cfg.patch_size}, native {Wn}x{Hn})", fontsize=12)
axes[0].imshow(img_disp);                            axes[0].set_title("Image native");           axes[0].axis("off")
axes[1].imshow(mask_np,   cmap="gray", vmin=0, vmax=1); axes[1].set_title("Points GT (5mm)");        axes[1].axis("off")
axes[2].imshow(pred_full, cmap="gray", vmin=0, vmax=1); axes[2].set_title("Prediction (tuiles)");     axes[2].axis("off")
axes[3].imshow(overlay);                             axes[3].set_title("Overlay (V=TP, R=FP, B=FN)"); axes[3].axis("off")
plt.tight_layout(); plt.show(); plt.close()
